# An edge-consistent RFQ responder

This notebook demonstrates, end to end, why a bond-dealer RFQ responder should price
the *conditional* value of winning rather than the unconditional value of the bond.
Everything below runs on synthetic data with a fixed seed. All models, objectives,
simulation logic, and plotting live in the `rfq_edge` package; the notebook only
imports functions, calls them, displays results, and interprets them.

## Section 1 — The question

When a dealer answers an RFQ it chooses a quote `q` for an observable RFQ state `X`.
A **plain responder** assumes that if it wins, the bond is still worth its pre-trade
estimate `V0`: winning is treated as luck.

The **edge-consistent responder** asks a sharper question:

> *"If I win specifically because I quoted `q`, what future clean value should I expect?"*

Clients sometimes know things dealers do not. A client who insists on selling often
sells ahead of bad news, so the dealer's fills are a *selected* sample. Three objects
capture this:

* `p(q, X)` — probability of winning at quote `q`;
* `m(q, X)` — expected future clean value *conditional on winning at `q`*;
* `J(q, X) = p(q, X) · [ side_sign · (m(q, X) − q) − cost + inventory value ]` — the expected objective.

The responder quotes the `q` that maximizes `J`, and declines when every candidate has `J ≤ 0`.

In [ ]:
import pandas as pd

from rfq_edge import plots
from rfq_edge.config import OptimizerConfig
from rfq_edge.evaluation import liquidity_buckets
from rfq_edge.fill_model import (
    counterfactual_fill_curve,
    counterfactual_fill_curves_by,
    evaluate_fill_model,
)
from rfq_edge.pipeline import cold_start_comparison, fit_framework, score_rfq
from rfq_edge.policy_evaluation import evaluate_policies, run_sensitivity_analysis
from rfq_edge.responders import compare_responders, observable_view
from rfq_edge.selection_model import (
    counterfactual_selection_curve,
    evaluate_selection_model,
    format_selection_metrics,
    make_selection_target,
    predict_selection,
    predicted_selection_by_liquidity,
)
from rfq_edge.simulation_diagnostics import (
    append_oracle_objective,
    build_oracle_context,
    oracle_best_decision,
    oracle_optimal_quote,
    oracle_selection,
    post_win_tilt_by_side,
    realized_selection_summary,
    win_rate_by_aggressiveness_bucket,
)
from rfq_edge.synthetic import SyntheticConfig, make_synthetic_rfqs, validate_synthetic_data
from rfq_edge.value_model import evaluate_value_models

SEED = 42
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## Section 2 — How the synthetic RFQ market works

The simulator has an **observable path** that fitted models may use, and a **hidden path**
that only the simulation (and its oracle diagnostics) may touch.

In [ ]:
fig, ax = plots.plot_simulation_workflow()

The top row is what a production system would see: the RFQ state `X`, the dealer's quote,
the client's decision, and an independently marked t+5 clean value `Y5`. The bottom row is
the hidden mechanism: latent client information is correlated with `Y5` and shifts the
client's decision, which is exactly what makes *winning informative*. Latent variables are
stored in `latent_*` columns; the feature layer raises an error if any model ever
receives one.

## Section 3 — Generate the market

We generate one year of synthetic RFQ flow with a fixed seed. Latent columns are kept on
the dataframe for oracle diagnostics only; `observable_view` strips them before anything
touches a model.

In [ ]:
synthetic_config = SyntheticConfig()
market = make_synthetic_rfqs(config=synthetic_config, random_state=SEED, include_latent=True)
observable = observable_view(market)
summary = validate_synthetic_data(observable)
pd.Series({key: value for key, value in summary.items() if not isinstance(value, dict)}).round(3)

In [ ]:
print("Date range:", observable["timestamp"].min().date(), "to", observable["timestamp"].max().date())
print()
print("Side balance:")
print(observable["side"].value_counts(normalize=True).round(3).to_string())
print()
print("Rating distribution:")
print(observable["rating_bucket"].value_counts(normalize=True).round(3).to_string())
print()
print("Regime distribution:")
print(observable["regime"].value_counts(normalize=True).round(3).to_string())
print()
print("Client tier distribution:")
print(observable["client_tier"].value_counts(normalize=True).round(3).to_string())
print()
print("Liquidity buckets:")
print(liquidity_buckets(observable["liquidity_score"]).value_counts(normalize=True).round(3).to_string())

In [ ]:
fig, ax = plots.plot_bond_activity_distribution(observable)

In [ ]:
fig, ax = plots.plot_synthetic_price_paths(observable, n_bonds=5)

In [ ]:
fig, ax = plots.plot_distribution(
    observable["market_width"] * 100.0,
    title="Market width distribution",
    xlabel="Market width (cents)",
)
fig, ax = plots.plot_distribution(
    observable["size"],
    title="RFQ size distribution (log scale)",
    xlabel="RFQ size (notional units)",
    log_x=True,
)

The market has ~15,000 RFQs across 300 bonds and 60 issuers, but activity is heavily
skewed: the median bond sees only a handful of RFQs while a few names dominate flow.
**Per-bond models are impractical** — most bonds have too little history to estimate
anything. Instead, all models pool across bonds and issuers with categorical encodings
that shrink sparse names toward issuer- and population-level behavior.

## Section 4 — Prove that the simulation contains adverse selection

Before fitting anything, we verify the economics we claim to model. These diagnostics use
realized outcomes (and are the only place outside oracle evaluation where the latent
mechanism is discussed); fitted models never see any of it.

In [ ]:
buckets = win_rate_by_aggressiveness_bucket(observable)
fig, ax = plots.plot_empirical_win_rate_by_aggressiveness(buckets)

In [ ]:
fig, axes = plots.plot_hidden_information_mechanism(observable)

In [ ]:
tilt = post_win_tilt_by_side(observable)
print(tilt.round(4).to_string(index=False))
print()
selection_summary = realized_selection_summary(observable)
print(pd.Series(selection_summary).round(4).to_string())

Three facts confirm adverse selection is present:
win probability rises monotonically with quote aggressiveness; **dealer-buy wins tilt
toward lower subsequent values while dealer-sell wins tilt toward higher ones** (the red
win distributions shift in opposite directions per side); and realized selection measured
against CP+ is clearly worse among wins than in the full population. Winning is not luck —
it is information.

## Section 5 — Chronological train/test split

`fit_framework` fits everything on the earlier 80% of RFQs and holds out the final 20%.
Out-of-fold V0 predictions and all hyper-parameter searches use expanding-window
(chronological) cross-validation inside the training period only.

In [ ]:
framework = fit_framework(market)
train_end = framework.train_df["timestamp"].max()
print(f"Train: {len(framework.train_df):,} RFQs through {pd.to_datetime(train_end).date()}")
print(f"Test:  {len(framework.test_df):,} RFQs strictly after the boundary")
fig, ax = plots.plot_chronological_split(observable["timestamp"], train_end)

A random split would be misleading for two reasons. First, prices are temporally
dependent: a random split leaks the future into training through same-day and same-week
rows on the same bond. Second, the deployment question is inherently chronological — the
desk must quote tomorrow with models trained on yesterday. Validation folds inside the
training window (blue) tune hyper-parameters; the red test period is touched exactly once.

## Section 6 — Learn V0, the pre-action value

We compare three forecasts of the t+5 clean price on the held-out period: the CP+
composite alone, the raw internal mid, and the regularized pooled V0 model.

In [ ]:
fig, ax = plots.plot_internal_signal_vs_future_value(observable)

The internal mid deviates from CP+ in a way that is positively correlated with the
realized future residual, so it carries genuine signal — but the cloud around the
regression line shows it is noisy and biased in places. This is precisely the setting
where a regularized pooled model should beat both raw anchors: it can keep the useful
part of the internal signal while shrinking its noise toward CP+.

In [ ]:
train_obs = observable_view(framework.train_df)
test_obs = observable_view(framework.test_df)
value_eval = evaluate_value_models(train_obs, test_obs)
print(value_eval["metrics_table"])

In [ ]:
fig, ax = plots.plot_value_model_comparison(value_eval["metrics_by_model"])
v0_forecast = value_eval["forecasts"]["regularized"]
fig, ax = plots.plot_value_prediction_calibration(
    v0_forecast - test_obs["cp_plus"],
    test_obs["y5"] - test_obs["cp_plus"],
)

In [ ]:
fig, ax = plots.plot_value_residuals_over_time(test_obs["timestamp"], v0_forecast - test_obs["y5"])
mae_by_regime = {
    name: {regime: mae * 100.0 for regime, mae in metrics["by_regime"].items()}
    for name, metrics in value_eval["metrics_by_model"].items()
}
fig, ax = plots.plot_value_performance_by_regime(mae_by_regime)
print("MAE (points) by bond-history bucket:")
print(
    pd.DataFrame(
        {name: metrics["by_bond_history_bucket"] for name, metrics in value_eval["metrics_by_model"].items()}
    ).round(4).to_string()
)

The pooled V0 model beats both CP+ and the raw internal mid on MAE and size-weighted MAE,
its bucket-mean calibration hugs the diagonal, and the advantage persists through time,
across regimes (all models degrade in volatile weeks, as they should), and across
bond-history buckets. **Important framing:** this section tests *future-value prediction
only*. A better V0 does not by itself prove better RFQ execution — quoting changes which
trades you win, which is the subject of the rest of the notebook.

## Section 7 — Learn fill probability

We fit `p(q, X) = P(win | q, X)` on all training RFQs, wins and losses. The quote enters
only through normalized aggressiveness `z = side_sign · (q − CP+) / width`, so the model
supports counterfactual quotes.

In [ ]:
fill_eval = evaluate_fill_model(train_obs, test_obs)
fig, ax = plots.plot_fill_calibration(
    fill_eval["calibration_curve"], fill_eval["brier_score"], fill_eval["log_loss"]
)

In [ ]:
fill_curve = counterfactual_fill_curve(framework.models.fill_model, test_obs)
empirical = win_rate_by_aggressiveness_bucket(test_obs)
fig, ax = plots.plot_fill_probability_by_aggressiveness(fill_curve, empirical)
side_curves = counterfactual_fill_curves_by(framework.models.fill_model, test_obs, "side")
fig, ax = plots.plot_fill_probability_by_side(side_curves)

In [ ]:
tier_curves = counterfactual_fill_curves_by(framework.models.fill_model, test_obs, "client_tier")
fig, ax = plots.plot_fill_probability_by_client_tier(tier_curves)
liquidity_frame = test_obs.assign(liquidity_bucket=liquidity_buckets(test_obs["liquidity_score"]))
liquidity_curves = counterfactual_fill_curves_by(
    framework.models.fill_model, liquidity_frame, "liquidity_bucket"
)
fig, ax = plots.plot_fill_probability_by_client_tier(
    liquidity_curves, title="Fill probability by liquidity bucket"
)

Calibration is close to the diagonal on the held-out period, and the counterfactual curve
confirms the crucial monotonicity: **larger normalized aggressiveness raises win
probability**, on both dealer sides. Client tiers and liquidity buckets shift the level of
the curve — informed tiers and illiquid names are systematically harder or easier to win —
which is exactly the pooled structure the optimizer needs.

## Section 8 — Learn adverse selection

On won RFQs we build the realized selection target `D = side_sign · (V0_oof − Y5)` using
*out-of-fold* V0 so the target contains no training leak. We fit
`A(q, X) = E[D | win, q, X]` on fills, then form the conditional mark
`m(q, X) = V0 − side_sign · A(q, X)`.

In [ ]:
train_fills = train_obs.loc[train_obs["won"] & train_obs["v0_oof"].notna()]
test_fills = test_obs.loc[test_obs["won"] & test_obs["v0_oof"].notna()]
selection_eval = evaluate_selection_model(train_fills, test_fills)
print(format_selection_metrics(selection_eval))

In [ ]:
selection_model = framework.models.selection_model
predicted_selection = predict_selection(selection_model, test_fills)
realized_selection = make_selection_target(test_fills)
fig, ax = plots.plot_selection_calibration(predicted_selection, realized_selection)

In [ ]:
oracle_context = build_oracle_context(market, synthetic_config)
latent_test_fills = framework.test_df.loc[
    framework.test_df["won"] & framework.test_df["v0_oof"].notna()
]
oracle_sample = latent_test_fills.iloc[:400]
oracle_sel = oracle_selection(oracle_sample, oracle_sample["quote"], oracle_context, n_draws=1000)
predicted_on_sample = predict_selection(selection_model, observable_view(oracle_sample))
fig, ax = plots.plot_predicted_vs_oracle_selection(predicted_on_sample, oracle_sel)

In [ ]:
selection_curve = counterfactual_selection_curve(selection_model, test_fills)
fig, ax = plots.plot_selection_by_aggressiveness(selection_curve)
liquidity_selection = predicted_selection_by_liquidity(selection_model, test_fills)
fig, ax = plots.plot_selection_by_liquidity(liquidity_selection)
print("Predicted selection by client tier (cents):")
print((predicted_selection.groupby(test_fills["client_tier"]).mean() * 100.0).round(2).to_string())

Predicted selection tracks both the realized target and the oracle counterfactual at the
bucket level. Selection is *largest at passive quotes* — if you only win when a client
insists, the winner's curse is severe — and it concentrates in illiquid names and
informed client tiers. Sparse bonds borrow strength through pooled issuer, rating, tier,
and population one-hot levels (rare categories share a pooled "infrequent" level), so a
bond with three fills still receives a sensible estimate. The optimizer additionally
supports a **conservative haircut** multiplier on `A(q, X)` (`selection_haircut` in
`OptimizerConfig`), explored in Section 13.

## Section 9 — Follow one RFQ through the entire framework

We take one representative **dealer-buy** RFQ from the held-out set and score every
candidate quote on the shared grid.

In [ ]:
optimizer_config = OptimizerConfig()
buy_candidates = framework.test_df.loc[
    (framework.test_df["side"] == "dealer_buy") & framework.test_df["v0_oof"].notna()
]
rfq_buy = buy_candidates.iloc[[4]]
state_columns = [
    "cp_plus", "internal_mid", "v0_oof", "side", "size", "liquidity_score",
    "market_width", "client_tier", "inventory", "regime", "number_of_dealers",
    "quote_deadline_ms", "is_inventory_axe",
]
rfq_buy[state_columns].T

In [ ]:
scored_buy = score_rfq(framework, rfq_buy, optimizer_config)
grid_buy = scored_buy["grid"]
grid_columns = [
    "quote", "aggressiveness", "p_win", "selection_points",
    "post_win_value_edge_consistent", "apparent_edge_cents",
    "edge_cents_edge_consistent", "cost_cents", "inventory_value_cents",
    "expected_value_cents_edge_consistent", "in_support",
]
grid_buy[grid_columns].round(2)

In [ ]:
oracle_grid_buy = oracle_optimal_quote(rfq_buy, oracle_context, optimizer_config)
fig, axes = plots.plot_quote_surface(
    grid_buy, scored_buy["comparison"], oracle_grid_buy, side_label="(dealer buy)"
)

Reading the four panels for a dealer buy (quotes *below* CP+ are passive): fill
probability rises as the quote approaches the market; the conditional mark `m(q, X)` sits
*below* the unconditional `V0` because winning a buy selects sellers with bad news; the
conditional edge is therefore thinner than the apparent edge everywhere; and the expected
objective peaks at a less aggressive quote than the apparent-edge curves would suggest.
Now the same machinery on a **dealer-sell** RFQ, to prove the signs flip correctly.

In [ ]:
sell_candidates = framework.test_df.loc[
    (framework.test_df["side"] == "dealer_sell") & framework.test_df["v0_oof"].notna()
]
rfq_sell = sell_candidates.iloc[[0]]
scored_sell = score_rfq(framework, rfq_sell, optimizer_config)
oracle_grid_sell = oracle_optimal_quote(rfq_sell, oracle_context, optimizer_config)
fig, axes = plots.plot_quote_surface(
    scored_sell["grid"], scored_sell["comparison"], oracle_grid_sell, side_label="(dealer sell)"
)

For the sell RFQ everything mirrors: aggressive quotes are now *lower* prices, and the
conditional mark sits *above* `V0` — winning a sell means the buyer expected the price to
rise. In both directions `m(q, X)` moves against the dealer, which is the signature of
adverse selection handled with correct signs.

## Section 10 — Compare responders for the same RFQ

All three responders see the identical grid, fill model, costs, inventory values, quote
support, and decline option. They differ only in the post-win value they assume. The
oracle-optimal action (computed from the true simulator, diagnostics only) is shown for
reference. The visual comparison of the three objective curves — with each responder's
selected quote marked and the oracle truth dashed in black — is the bottom-right panel
of the quote surface in Section 9; the bar chart below compares the objectives at the
selected quotes.

In [ ]:
comparison_buy = append_oracle_objective(
    scored_buy["comparison"], rfq_buy, oracle_context, optimizer_config
)
oracle_best = oracle_best_decision(oracle_grid_buy)
print("Oracle-optimal action (simulation diagnostic only):")
print(pd.Series(oracle_best).round(3).to_string())
comparison_columns = [
    "responder", "accepted", "quote", "aggressiveness", "p_win", "selection_points",
    "conditional_edge_cents", "expected_value_cents", "oracle_expected_objective_cents",
]
comparison_buy[comparison_columns].round(3)

In [ ]:
fig, ax = plots.plot_responder_comparison(comparison_buy)

Why the selected quotes differ: the **plain CP+ responder** believes any purchase below
CP+ is profit, so it maximizes fill-weighted apparent edge against CP+ and selects the
most aggressive quote of the three. The **plain V0 responder** upgrades the value anchor
but still treats winning as uninformative, so it quotes one grid step more cautiously.
The **edge-consistent responder** additionally deducts the predicted post-win value
change; because selection is priced exactly where the plain responders see the most
apparent edge, its objective curve bends down asymmetrically and its optimum lands at
the most passive quote of the three. The oracle column shows how much of each
responder's *predicted* objective survives contact with the true conditional
distribution — the more aggressive quotes systematically over-promise. On RFQs where
even the best supported candidate has a nonpositive objective, every responder can and
does decline (Section 11 reports decline rates).

## Section 11 — Compare responders over the held-out test set

Every responder now quotes the *same* held-out RFQs. Realized fills are simulated from the
oracle's exact counterfactual fill probability using **common random numbers** (one
uniform draw per RFQ shared by all responders), and realized values use the dataset's
independent `Y5` marks. Confidence intervals come from bootstrapping whole *dates*, not
individual RFQs. **Simulated clean value is a diagnostic — it is not real trading PnL.**

In [ ]:
policy = evaluate_policies(
    framework.test_df,
    framework.models,
    oracle_context,
    optimizer_config,
    random_state=SEED,
)
policy.summary.round(3)

In [ ]:
print(policy.bootstrap.round(3).to_string(index=False))
fig, ax = plots.plot_policy_performance(policy.summary, policy.bootstrap)

In [ ]:
fig, ax = plots.plot_selected_quotes_distribution(policy.decisions)
fig, ax = plots.plot_quote_frontier(policy.summary)

In [ ]:
fig, ax = plots.plot_cumulative_clean_value(policy.decisions)
fig, ax = plots.plot_policy_heatmap(policy.decisions, responder_label="Edge-consistent")

In [ ]:
for segment_name in ("side", "liquidity_bucket", "rating_bucket", "regime"):
    print(f"--- net clean value per RFQ by {segment_name} ---")
    print(policy.segment_summaries[segment_name].round(3).to_string(index=False))
    print()

The edge-consistent responder systematically selects less aggressive quotes (its
distribution shifts left), fills slightly less often, and gives up apparent edge in
exchange for avoiding the worst-selected fills — visible as higher net clean value per
fill and a cumulative value line that separates most in illiquid names and volatile
regimes, where selection is strongest. The date-block bootstrap intervals overlap
substantially: on a single simulated year the ranking is suggestive, not decisive, and
that honesty is part of the point.

## Section 12 — Sparse and cold-start examples

How do predictions behave when history is thin? We compare an active bond, a sparse bond,
an unseen bond from a known issuer, and a bond from an entirely unseen issuer.

In [ ]:
cold_start = cold_start_comparison(framework, optimizer_config)
cold_start.round(3)

As bond history disappears, the encoders shrink predictions toward issuer-level and then
population-level behavior: the unseen bond inherits its issuer's profile, and the unseen
issuer receives close-to-population estimates. Decisions degrade gracefully instead of
failing — the framework quotes from pooled behavior with wider implicit uncertainty rather
than refusing or extrapolating from noise.

## Section 13 — Sensitivity analysis

We repeat the held-out policy comparison under alternative cost and correction
calibrations, reusing the same RFQs and the same random fill draws so changes are
attributable to the calibration alone. Transaction costs are expressed in cents on a
par-100 bond (1 bp of par = 1 cent).

In [ ]:
scenarios = {
    "baseline": OptimizerConfig(),
    "5c transaction cost": OptimizerConfig(transaction_bps=5.0),
    "7.5c transaction cost": OptimizerConfig(transaction_bps=7.5),
    "10c transaction cost": OptimizerConfig(transaction_bps=10.0),
    "no inventory effect": OptimizerConfig(
        inventory_value_per_unit=0.0, inventory_penalty_per_unit=0.0
    ),
    "strong inventory penalty": OptimizerConfig(inventory_penalty_per_unit=0.00006),
    "no selection correction": OptimizerConfig(selection_haircut=0.0),
    "conservative selection haircut": OptimizerConfig(selection_haircut=1.5),
}
sensitivity = run_sensitivity_analysis(
    framework.test_df, framework.models, oracle_context, scenarios, random_state=SEED
)
print("Net clean value per RFQ (cents):")
print(
    sensitivity.pivot(index="scenario", columns="responder", values="net_value_per_rfq_cents")
    .round(2).to_string()
)
print()
print("Decline rate:")
print(
    sensitivity.pivot(index="scenario", columns="responder", values="decline_rate")
    .round(3).to_string()
)

In [ ]:
fig, axes = plots.plot_sensitivity_analysis(sensitivity)

Higher transaction costs push all responders toward more passive quotes and higher
decline rates, but the edge-consistent responder declines *earlier* because its measured
edge is thinner and closer to the truth. Removing the selection correction
(`selection_haircut = 0`) collapses it onto the plain V0 responder, while the
conservative haircut trades fill rate for protection — a knob a desk would calibrate to
its own risk appetite.

## Section 14 — Conclusions and limitations

**What the simulation shows**

* Predictive value is not the same as executable RFQ edge: a better `V0` improved
  forecasts long before it changed quoting outcomes.
* Quoting changes the probability of winning — `p(q, X)` is steep in aggressiveness.
* Winning changes the conditional distribution of future value — fills are a selected
  sample, in opposite directions for dealer buys and sells.
* The correct decision quantity is the **conditional clean edge**
  `side_sign · (m(q, X) − q)`, combined with costs and inventory value inside `J(q, X)`.
* Sparse estimation requires pooling across bonds, issuers, and clients, plus
  conservative fallbacks (support restrictions, haircuts, and the decline option).

**Limitations to keep in mind**

* The data are synthetic and calibrated for illustration; magnitudes are not market
  estimates.
* Real observational data do not reveal counterfactual quote outcomes; here the oracle
  exists only because we own the simulator.
* Quote support matters: outside historically quoted aggressiveness, every model output
  is extrapolation, and the responders refuse to act there.
* Hidden confounding may remain in real data even after conditioning on rich RFQ state.
* The t+5 clean value is a marking convention, not cash PnL — no funding, hedge slippage,
  or balance-sheet effects are modelled.
* Costs and inventory values were set by hand; a production system needs empirical
  calibration.

The purpose of this demonstration is not to claim the edge-consistent responder always
wins. It is to show **when and why accounting for post-win value changes quote
decisions** — most visibly for informed clients, illiquid bonds, and volatile regimes,
where the plain responders systematically overpay for the trades they are most likely
to win.